# 03 Feature Extraction
**Purpose of this notebook:** convert each of the 400 texts into 11 linguistic and stylometric features grouped into five categories. These features are the only input to the classifier; the raw text is never passed to the model.

The feature set is fixed here, before any classification experiment is run.

**Input:** `all_texts.csv` (from 02)
**Output:** `features.csv` (400 rows, 11 features plus metadata)

## Step 1: Install and load tools
spaCy (`en_core_web_sm`) provides tokenisation, sentence segmentation and part-of-speech tagging. VADER provides the sentiment score.

In [ ]:
!pip install -q vaderSentiment
!python -m spacy download en_core_web_sm -q

In [ ]:
import re
import pandas as pd
import spacy
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

nlp = spacy.load("en_core_web_sm")
vader = SentimentIntensityAnalyzer()
print("spaCy", spacy.__version__)

## Step 2: Load the dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv("all_texts.csv")
print(len(df), "texts")
print(df["source"].value_counts())
assert len(df) == 400 and df["text"].isna().sum() == 0
df.head(2)

## Step 3: Feature definitions

| Group | Feature | Definition |
|---|---|---|
| Lexical | `ttr` | Type–token ratio over the **first 100 content tokens** (lowercased), so that the measure is independent of text length |
| Sentence | `avg_sent_len` | Content tokens divided by number of sentences |
| POS | `noun_ratio` | (NOUN + PROPN) tokens / content tokens |
| POS | `verb_ratio` | VERB tokens / content tokens |
| POS | `adj_ratio` | ADJ tokens / content tokens |
| POS | `adv_ratio` | ADV tokens / content tokens |
| POS | `pron_ratio` | PRON tokens / content tokens |
| Sentiment | `sent_intensity` | Absolute VADER compound score; the polarity itself is controlled by the paired design, so only its strength is informative |
| Punctuation | `comma_rate` | Commas per 100 content tokens |
| Punctuation | `excl_rate` | Exclamation marks per 100 content tokens |
| Punctuation | `quest_rate` | Question marks per 100 content tokens |

"Content tokens" excludes punctuation and whitespace throughout, so that punctuation density does not inflate the denominator of the ratio features. All rate features are normalised per 100 tokens, so that the residual length difference between human and generated texts cannot drive the comparison.

In [ ]:
FEATURES = ["ttr", "avg_sent_len",
            "noun_ratio", "verb_ratio", "adj_ratio", "adv_ratio", "pron_ratio",
            "sent_intensity",
            "comma_rate", "excl_rate", "quest_rate"]

TTR_WINDOW = 100

def extract(text):
    doc = nlp(text)
    toks = [t for t in doc if not t.is_punct and not t.is_space]
    n = len(toks)
    sents = [s for s in doc.sents if len(s.text.strip()) > 0]

    words = [t.text.lower() for t in toks][:TTR_WINDOW]
    ttr = len(set(words)) / len(words)

    pos = {}
    for tag in ["NOUN", "PROPN", "VERB", "ADJ", "ADV", "PRON"]:
        pos[tag] = sum(1 for t in toks if t.pos_ == tag)

    return {
        "n_tokens": n,
        "n_sents": len(sents),
        "ttr": ttr,
        "avg_sent_len": n / len(sents),
        "noun_ratio": (pos["NOUN"] + pos["PROPN"]) / n,
        "verb_ratio": pos["VERB"] / n,
        "adj_ratio": pos["ADJ"] / n,
        "adv_ratio": pos["ADV"] / n,
        "pron_ratio": pos["PRON"] / n,
        "sent_intensity": abs(vader.polarity_scores(text)["compound"]),
        "comma_rate": text.count(",") / n * 100,
        "excl_rate": text.count("!") / n * 100,
        "quest_rate": text.count("?") / n * 100,
    }

# 测试
demo = extract(df["text"].iloc[0])
for k, v in demo.items():
    print(f"{k:16s} {v:.3f}")

## Step 4: Apply to all texts

In [ ]:
from tqdm.auto import tqdm
tqdm.pandas()

feat = pd.DataFrame([extract(t) for t in tqdm(df["text"])])
meta = df[["text_id", "review_id", "tconst", "primaryTitle", "label",
           "llm_group", "source", "model", "is_llm", "word_count"]]
features = pd.concat([meta.reset_index(drop=True), feat], axis=1)
features.to_csv("features.csv", index=False)
print(features.shape)
features.head(3)

## Step 5: Verification
Confirms that every text produced a complete feature vector and that all values fall in a plausible range.

In [ ]:
print("Rows:", len(features))
print("Missing values:", features[FEATURES].isna().sum().sum())
import numpy as np
print("Infinite values:", np.isinf(features[FEATURES].to_numpy()).sum())
print()
print(features[FEATURES].describe().T[["min", "mean", "max"]].round(3))
print()
print("Ratio features sum <= 1:", (features[["noun_ratio","verb_ratio","adj_ratio","adv_ratio","pron_ratio"]].sum(axis=1) <= 1).all())
print("TTR within (0,1]:", features["ttr"].between(0, 1, inclusive="right").all())

## Step 6: Descriptive comparison (preview of E1)
Group means per source. The formal statistical tests are run in 04; this table is only a first look at whether the three groups differ.

In [ ]:
summary = features.groupby("source")[FEATURES].mean().round(3).T
summary["human_vs_llmA"] = (summary["llm_A"] - summary["human"]).round(3)
summary["human_vs_llmB"] = (summary["llm_B"] - summary["human"]).round(3)
print(summary)

In [ ]:
files.download("features.csv")